In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [ ]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [ ]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [ ]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

In [ ]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

In [ ]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

In [ ]:
def get_thinking_trace_from_completion(completion: str) -> str:
    prefix_split_str = '<|im_start|>assistant\n'
    completion_with_suffix = completion.split(prefix_split_str)[-1].strip()

    suffix_split_str = '<|im_end|>'
    completion = completion_with_suffix.split(suffix_split_str)[0].strip()
    return completion


def get_token_length(text: str, tokenizer: PreTrainedTokenizer) -> int:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    return len(token_ids)

In [ ]:
df['partial_trace'] = df['pivotal_context'].apply(get_thinking_trace_from_completion)
df['partial_trace_token_length'] = df['partial_trace'].apply(lambda x: get_token_length(x, tokenizer))
df['trace_token_length'] = df['trace'].apply(lambda x: get_token_length(x, tokenizer))
df['pivotal_token_relative_position'] = (df['partial_trace_token_length'] + 1) / df['trace_token_length']

assert (df['partial_trace_token_length'] < df['trace_token_length']).mean() == 1.0

In [ ]:
def get_init_success_prob(sample_id: str, base_repo: DictRepo) -> float | None:
    repo = SampleRepo(base_repo=base_repo, sample_id=sample_id)
    subdivisions = repo.list(path="subdivisions")

    init_prob = None
    for subdiv in subdivisions:
        data = repo.load(path="subdivisions", key=subdiv)
        if len(data['prefix']) == 0 and \
                data['full_seq'].startswith('<think>') and \
                data['full_seq'].endswith('</think>'):
            init_prob = data['prob_before']
            break
    
    return init_prob

In [ ]:
df['prob_init'] = df['sample_id'].apply(lambda x: get_init_success_prob(x, base_repo))

In [ ]:
df['prob_before_normalized'] = df['prob_before'] / df['prob_init']
df['prob_after_normalized'] = df['prob_after'] / df['prob_init']
df['prob_delta_normalized'] = df['prob_delta'] / df['prob_init']

In [ ]:
df['is_positive'] = df['prob_delta'] >= 0

In [ ]:
df = df[df['is_pivotal'] == True].copy()

* * *

In [ ]:
print(f"Number of pivotal tokens: {len(df)}")

df['is_positive'].value_counts()

In [ ]:
# df = df[df['is_positive'] == True].copy()

In [ ]:
counts = df.groupby('sample_id', as_index=False).agg({'span_id': 'nunique'})
px.histogram(counts, x='span_id', nbins=20, title='Number of Positive Pivotal Tokens per Sample').show()

In [ ]:
# Check relation between pivotal token position and trace length
fig = px.scatter(df,
                x='trace_token_length',
                y='partial_trace_token_length',
                hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                title='Partial Trace Token Length vs Full Trace Token Length')
fig.update_layout(width=700)
fig.show()

In [ ]:
# Check if longer traces correlate with lower initial success probability
fig = px.scatter(df,
                 x='trace_token_length' ,
                 y='prob_init',
                 hover_data=['sample_id', 'prob_before', 'prob_after'],
                 title='Initial Success Probability vs Full Trace Token Length'
                 )
fig.update_layout(width=700)
fig.show()

In [ ]:
# Check relation between pivotal token position and its impact on success probability
fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()

In [ ]:
# Check relation between pivotal token position and its normalized impact on success probability
from turtle import width


fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta_normalized',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta_normalized',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()

In [ ]:
# Check relation between pivotal token position and trace length
fig = px.scatter(df,
                x='prob_delta',
                y='prob_init',
                marginal_y='histogram',
                hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                title='Initial Success Probability vs Probability Delta')
fig.update_layout(width=700)
fig.show()

In [ ]:
fig = px.histogram(df, x='prob_delta', nbins=50, title='Histogram of Probability Delta')
fig.show()

In [ ]:
fig = px.histogram(df, x='prob_delta_normalized', nbins=50, title='Histogram of Normalized Probability Delta')
fig.show()

* * *

# Optimal threshold selection

In [ ]:
import numpy as np
import scipy.stats as stats
from sklearn.mixture import GaussianMixture
import plotly.graph_objects as go
from plotly.subplots import make_subplots

raw_df = tokens_df.copy()
raw_df['abs_delta'] = raw_df['prob_delta'].abs()
df['abs_delta'] = df['prob_delta'].abs()
df['abs_delta_norm'] = df['prob_delta_normalized'].abs()
total_samples_raw = raw_df['sample_id'].nunique()
total_samples_df = df['sample_id'].nunique()

print(f"All tokens (no filter): {len(raw_df)}, samples: {total_samples_raw}")
print(f"is_pivotal=True: {raw_df['is_pivotal'].sum()}, is_pivotal=False: {(~raw_df['is_pivotal']).sum()}")
print(f"|prob_delta| range: [{raw_df['abs_delta'].min():.4f}, {raw_df['abs_delta'].max():.4f}]")


def build_sweep(source_df, delta_col, extra_cols=None, n_steps=100, threshold_col='threshold'):
    """Threshold sweep: for each T, compute n_tokens, coverage, and median deltas."""
    deltas = source_df[delta_col].values
    sample_ids = source_df['sample_id'].values
    total = source_df['sample_id'].nunique()
    order = np.argsort(deltas)[::-1]
    s_delta = deltas[order]
    s_ids = sample_ids[order]
    extra = {c: source_df[c].values[order] for c in (extra_cols or [])}

    thresholds = np.linspace(0.0, np.quantile(deltas, 0.99), n_steps)
    rows = []
    for T in thresholds:
        cut = int(np.searchsorted(-s_delta, -T))
        row = {
            threshold_col: T,
            'n_tokens': cut,
            'coverage': len(np.unique(s_ids[:cut])) / total if cut > 0 else 0.0,
            f'median_{delta_col}': float(np.median(s_delta[:cut])) if cut > 0 else float('nan'),
        }
        for c, vals in extra.items():
            row[f'median_{c}'] = float(np.median(vals[:cut])) if cut > 0 else float('nan')
        rows.append(row)
    return pd.DataFrame(rows)


def fit_gmm(values):
    """Fit 2-component GMM; return (threshold, [(mu, sigma, weight), ...], x_range)."""
    X = values.reshape(-1, 1)
    gmm = GaussianMixture(n_components=2, random_state=42).fit(X)
    means = gmm.means_.flatten()
    stds = np.sqrt(gmm.covariances_.flatten())
    weights = gmm.weights_
    order = np.argsort(means)
    comps = [(means[i], stds[i], weights[i]) for i in order]
    mu1, s1, w1 = comps[0]
    mu2, s2, w2 = comps[1]
    x_range = np.linspace(0, values.max(), 2000)
    diff = w1 * stats.norm.pdf(x_range, mu1, s1) - w2 * stats.norm.pdf(x_range, mu2, s2)
    cross = x_range > mu1
    sc = np.where(np.diff(np.sign(diff[cross])))[0]
    threshold = x_range[cross][sc[0]] if len(sc) else (mu1 + mu2) / 2
    return threshold, comps, x_range


def detect_elbow(values):
    """Geometric elbow on sorted-descending values. Returns (sorted_values, elbow_idx, elbow_value)."""
    s = np.sort(values)[::-1]
    idx = np.arange(len(s), dtype=float)
    v = np.array([idx[-1] - 0.0, s[-1] - s[0]])
    # Scalar 2D cross product: v[0]*(s[0]-s[i]) - v[1]*(0-idx[i])
    cross = v[0] * (s[0] - s) - v[1] * (0.0 - idx)
    dists = np.abs(cross) / np.linalg.norm(v)
    ei = int(np.argmax(dists))
    return s, ei, float(s[ei])


def n_tokens_at(sweep_df, threshold_col, threshold_val):
    return int(sweep_df.loc[(sweep_df[threshold_col] - threshold_val).abs().idxmin(), 'n_tokens'])


def plot_gmm_hist(source_df, delta_col, comps, x_range, threshold, title):
    fig = px.histogram(source_df, x=delta_col, nbins=80, title=title, histnorm='probability density')
    for label, (mu, s, w), color in zip(['noise', 'signal'], comps, ['blue', 'green']):
        fig.add_scatter(x=x_range, y=w * stats.norm.pdf(x_range, mu, s), name=f'GMM {label}', line_color=color)
    fig.add_vline(x=threshold, line_dash='dash', line_color='red', annotation_text=f'GMM={threshold:.3f}')
    fig.show()


def plot_median(sweep_df, n_col, y_col, title, y_label, vline_x=None, vline_label=None, y_range=None, color=None):
    fig = go.Figure()
    trace_kwargs = dict(name=y_label)
    if color:
        trace_kwargs['line_color'] = color
    fig.add_trace(go.Scatter(x=sweep_df[n_col], y=sweep_df[y_col], **trace_kwargs))
    if vline_x is not None:
        fig.add_vline(x=vline_x, line_dash='dash', line_color='red', annotation_text=vline_label or '')
    layout = dict(title=title, xaxis_title='# tokens remaining', yaxis_title=y_label)
    if y_range:
        layout['yaxis_range'] = y_range
    fig.update_layout(**layout)
    fig.show()

In [ ]:
# sweep_df: threshold in prob_delta space; also tracks median_abs_delta_norm on the same filtered set
sweep_df = build_sweep(raw_df, 'abs_delta', extra_cols=None, threshold_col='threshold')
# Add median_abs_delta_norm: for each prob_delta threshold, median normalized delta of pivotal tokens passing it
# (uses df filtered by abs_delta, so units are consistent with sweep_df's population concept)
pivot_order = np.argsort(df['abs_delta'].values)[::-1]
pivot_s_delta = df['abs_delta'].values[pivot_order]
pivot_s_norm  = df['abs_delta_norm'].values[pivot_order]
sweep_df['median_abs_delta_norm'] = [
    float(np.median(pivot_s_norm[:int(np.searchsorted(-pivot_s_delta, -T))])) if int(np.searchsorted(-pivot_s_delta, -T)) > 0 else float('nan')
    for T in sweep_df['threshold']
]

# norm_sweep_df: threshold in prob_delta_normalized space
norm_sweep_df = build_sweep(df, 'abs_delta_norm', threshold_col='threshold_norm')

# Count & coverage
fig_sweep = make_subplots(specs=[[{"secondary_y": True}]])
fig_sweep.add_trace(go.Scatter(x=sweep_df['threshold'], y=sweep_df['n_tokens'], name='# tokens'), secondary_y=False)
fig_sweep.add_trace(go.Scatter(x=sweep_df['threshold'], y=sweep_df['coverage'], name='Sample coverage', line_color='orange'), secondary_y=True)
fig_sweep.update_layout(title='Threshold sweep: token count & sample coverage vs |prob_delta|', xaxis_title='|prob_delta| threshold')
fig_sweep.update_yaxes(title_text='# tokens', secondary_y=False)
fig_sweep.update_yaxes(title_text='Fraction of samples with ≥1 token', range=[0, 1], secondary_y=True)
fig_sweep.show()

plot_median(sweep_df, 'n_tokens', 'median_abs_delta', '# tokens remaining vs median |prob_delta|', 'median |prob_delta|', y_range=[0, 1])
plot_median(norm_sweep_df, 'n_tokens', 'median_abs_delta_norm', '# tokens remaining vs median |prob_delta_normalized|', 'median |prob_delta_normalized|', color='green')

In [ ]:
gmm_threshold, gmm_comps, gmm_x = fit_gmm(raw_df['abs_delta'].values)
(mu1, s1, w1), (mu2, s2, w2) = gmm_comps
print(f"GMM threshold (prob_delta): {gmm_threshold:.4f}")
print(f"  noise:  mu={mu1:.4f}, sigma={s1:.4f}, weight={w1:.3f}")
print(f"  signal: mu={mu2:.4f}, sigma={s2:.4f}, weight={w2:.3f}")
plot_gmm_hist(raw_df, 'abs_delta', gmm_comps, gmm_x, gmm_threshold, '|prob_delta| distribution with GMM components')

In [ ]:
n = n_tokens_at(sweep_df, 'threshold', gmm_threshold)
plot_median(sweep_df, 'n_tokens', 'median_abs_delta', 'GMM: # tokens remaining vs median |prob_delta|', 'median |prob_delta|', vline_x=n, vline_label=f'GMM ({n} tokens)', y_range=[0, 1])
plot_median(sweep_df, 'n_tokens', 'median_abs_delta_norm', 'GMM: # tokens remaining vs median |prob_delta_normalized|', 'median |prob_delta_normalized|', vline_x=n, vline_label=f'GMM ({n} tokens)', color='green')

In [ ]:
sorted_deltas, elbow_idx, elbow_threshold = detect_elbow(raw_df['abs_delta'].values)
print(f"Elbow threshold (prob_delta): {elbow_threshold:.4f} (rank {elbow_idx} of {len(sorted_deltas)})")

elbow_fig = go.Figure()
elbow_fig.add_trace(go.Scatter(x=np.arange(len(sorted_deltas)), y=sorted_deltas, name='|prob_delta|'))
elbow_fig.add_vline(x=elbow_idx, line_dash='dash', line_color='red', annotation_text=f'elbow={elbow_threshold:.3f}')
elbow_fig.update_layout(title='Sorted |prob_delta| — elbow detection', xaxis_title='Token rank (descending)', yaxis_title='|prob_delta|')
elbow_fig.show()

In [ ]:
n = n_tokens_at(sweep_df, 'threshold', elbow_threshold)
plot_median(sweep_df, 'n_tokens', 'median_abs_delta', 'Elbow: # tokens remaining vs median |prob_delta|', 'median |prob_delta|', vline_x=n, vline_label=f'Elbow ({n} tokens)', y_range=[0, 1])
plot_median(sweep_df, 'n_tokens', 'median_abs_delta_norm', 'Elbow: # tokens remaining vs median |prob_delta_normalized|', 'median |prob_delta_normalized|', vline_x=n, vline_label=f'Elbow ({n} tokens)', color='green')

### Normalized delta (prob_delta / prob_init)
Using pivotal-filtered `df` which already has `prob_delta_normalized` computed.

In [ ]:
gmm_threshold_norm, gmm_norm_comps, gmm_norm_x = fit_gmm(df['abs_delta_norm'].values)
(mu1n, s1n, w1n), (mu2n, s2n, w2n) = gmm_norm_comps
print(f"GMM threshold (prob_delta_normalized): {gmm_threshold_norm:.4f}")
print(f"  noise:  mu={mu1n:.4f}, sigma={s1n:.4f}, weight={w1n:.3f}")
print(f"  signal: mu={mu2n:.4f}, sigma={s2n:.4f}, weight={w2n:.3f}")
plot_gmm_hist(df, 'abs_delta_norm', gmm_norm_comps, gmm_norm_x, gmm_threshold_norm, '|prob_delta_normalized| distribution with GMM components')

In [ ]:
n = n_tokens_at(norm_sweep_df, 'threshold_norm', gmm_threshold_norm)
plot_median(norm_sweep_df, 'n_tokens', 'median_abs_delta_norm', 'GMM (normalized): # tokens remaining vs median |prob_delta_normalized|', 'median |prob_delta_normalized|', vline_x=n, vline_label=f'GMM ({n} tokens)', color='green')

In [ ]:
sorted_norm, elbow_idx_n, elbow_threshold_norm = detect_elbow(df['abs_delta_norm'].values)
print(f"Elbow threshold (prob_delta_normalized): {elbow_threshold_norm:.4f} (rank {elbow_idx_n} of {len(sorted_norm)})")

elbow_norm_fig = go.Figure()
elbow_norm_fig.add_trace(go.Scatter(x=np.arange(len(sorted_norm)), y=sorted_norm, name='|prob_delta_normalized|'))
elbow_norm_fig.add_vline(x=elbow_idx_n, line_dash='dash', line_color='red', annotation_text=f'elbow={elbow_threshold_norm:.3f}')
elbow_norm_fig.update_layout(title='Sorted |prob_delta_normalized| — elbow detection', xaxis_title='Token rank (descending)', yaxis_title='|prob_delta_normalized|')
elbow_norm_fig.show()

In [ ]:
n = n_tokens_at(norm_sweep_df, 'threshold_norm', elbow_threshold_norm)
plot_median(norm_sweep_df, 'n_tokens', 'median_abs_delta_norm', 'Elbow (normalized): # tokens remaining vs median |prob_delta_normalized|', 'median |prob_delta_normalized|', vline_x=n, vline_label=f'Elbow ({n} tokens)', color='green')

In [ ]:
from IPython.display import HTML, display

def threshold_stats(source_df, delta_col, threshold, total_samp):
    mask = source_df[delta_col] >= threshold
    n_tok = int(mask.sum())
    cov = source_df.loc[mask, 'sample_id'].nunique() / total_samp
    return n_tok, round(cov, 4)

rows = [
    ('Extraction threshold (exp2.1.2)', 'prob_delta', 0.05,
     *threshold_stats(raw_df, 'abs_delta', 0.05, total_samples_raw)),
    ('GMM', 'prob_delta', round(gmm_threshold, 4),
     *threshold_stats(raw_df, 'abs_delta', gmm_threshold, total_samples_raw)),
    ('Elbow', 'prob_delta', round(elbow_threshold, 4),
     *threshold_stats(raw_df, 'abs_delta', elbow_threshold, total_samples_raw)),
    ('GMM', 'prob_delta_normalized', round(gmm_threshold_norm, 4),
     *threshold_stats(df, 'abs_delta_norm', gmm_threshold_norm, total_samples_df)),
    ('Elbow', 'prob_delta_normalized', round(elbow_threshold_norm, 4),
     *threshold_stats(df, 'abs_delta_norm', elbow_threshold_norm, total_samples_df)),
]

summary_df = pd.DataFrame(rows, columns=['method', 'metric', 'threshold', 'n_tokens', 'coverage'])
display(HTML(summary_df.to_html(index=False)))